# 05 — Post-processing experiments

Raw model output vs harmonic filtering, velocity-ratio sweeps, and MusicXML quantization.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "tests").exists():
    ROOT = ROOT.parent
AUDIO_DIR = ROOT / "tests" / "fixtures" / "audio"
GOLDEN_DIR = ROOT / "tests" / "fixtures" / "golden"
print("project root:", ROOT)


project root: /home/anton/Projects/music-teacher


In [2]:
from music_teacher.transcription.audio import load_audio
from music_teacher.transcription.onnx_engine import BasicPitchEngine
from music_teacher.transcription.notes import filter_harmonic_duplicates, model_output_to_notes

engine = BasicPitchEngine(device="auto"); engine.warmup()
output = engine.predict(load_audio(AUDIO_DIR / "piano_chords.wav"))
raw = model_output_to_notes(output)
print("raw notes:", len(raw))
for event in raw:
    print(f"  {event.start:5.2f}-{event.end:5.2f} p={event.pitch:3d} v={event.velocity:.2f}")


raw notes: 15
   0.09- 0.30 p= 79 v=0.32
   0.10- 1.11 p= 60 v=0.78
   0.10- 1.11 p= 64 v=0.80
   0.10- 1.13 p= 67 v=0.73
   1.20- 2.22 p= 65 v=0.78
   1.20- 2.22 p= 72 v=0.73
   1.20- 1.35 p= 84 v=0.31
   1.21- 2.22 p= 69 v=0.82
   1.88- 2.15 p= 84 v=0.34
   2.29- 2.52 p= 86 v=0.37
   2.29- 2.92 p= 93 v=0.31
   2.30- 3.34 p= 67 v=0.76
   2.30- 3.33 p= 71 v=0.79
   2.30- 3.33 p= 74 v=0.72
   2.82- 3.23 p= 86 v=0.39


In [3]:
for ratio in (0.4, 0.5, 0.65, 0.8, 0.95):
    filtered = filter_harmonic_duplicates(raw, velocity_ratio=ratio)
    print(f"velocity_ratio={ratio}: {len(raw)} -> {len(filtered)} notes")


velocity_ratio=0.4: 15 -> 14 notes
velocity_ratio=0.5: 15 -> 10 notes
velocity_ratio=0.65: 15 -> 9 notes
velocity_ratio=0.8: 15 -> 9 notes
velocity_ratio=0.95: 15 -> 9 notes


In [4]:
from music_teacher.theory import musicxml_bytes
filtered = filter_harmonic_duplicates(raw)
xml = musicxml_bytes(filtered, tempo_bpm=120, title="piano_chords")
print(xml.decode()[:600])


<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 4.0 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">
<score-partwise version="4.0">
  <work>
    <work-title>piano_chords</work-title>
  </work>
  <part-list>
    <score-part id="P1">
      <part-name>Music Teacher</part-name>
    </score-part>
  </part-list>
  <part id="P1">
    <measure number="1">
      <attributes>
        <divisions>4</divisions>
        <key>
          <fifths>0</fifths>
        </key>
        <time>
          <beats>4</beats>
          <beat-type>4</beat-type>
  


The `filter_harmonic_duplicates` rule lives in `src/music_teacher/transcription/notes.py`. Tune it here, then add a test in `tests/test_notes.py`.